# PokouAI — Fine-tune Gemma 4 on Cocoa Diseases

QLoRA fine-tune of Gemma 4 (E2B primary / E4B premium) on cocoa pod images across 6 disease classes and 4 languages, exported as GGUF Q4_K_M for on-device inference.

- **Platform**: Kaggle T4 x2 (Unsloth uses one T4)
- **Stack**: Unsloth + PEFT + TRL
- **v2 scope**: 40k train / 1k val / 3 epochs ≈ 20–21 h on T4. Splits across 2 Kaggle sessions via `RESUME_FROM_CHECKPOINT` in §5; `save_steps=200` means at most ~3 min of wasted work if the first session dies. v1's 5k × 1 epoch was undercooked (hallucinated outputs, format not enforced).

## 1 — Environment setup

In [ ]:
# Install order matters:
# 1. Force-reinstall unsloth + unsloth_zoo from git HEAD together. Kaggle's
#    base image ships an old unsloth_zoo; the new unsloth from the [kaggle-new]
#    extra imports symbols only present in newer unsloth_zoo, so leaving the
#    cached unsloth_zoo in place yields:
#       ImportError: cannot import name '_unsloth_get_mm_token_id'
#       from 'unsloth_zoo.rl_replacements'
#    --no-deps prevents pip from "helpfully" downgrading either side to satisfy
#    the resolver.
# 2. Standard transformers/peft/trl stack. unsloth's [kaggle-new] extra pulls
#    these too, but pinning our own ensures compatible versions.
# 3. torchao upgrade. Kaggle base image ships torchao 0.10.0; current PEFT
#    requires >= 0.16.0 or get_peft_model() raises:
#       ImportError: Found an incompatible version of torchao. Found 0.10.0,
#       but only versions above 0.16.0 are supported
# 4. Pillow ABI pin LAST. Anything installing after this can pull Pillow 12.x
#    back in, which mismatches Kaggle's compiled _imaging.so (11.3).

!pip install -qU --force-reinstall --no-deps \
    "unsloth @ git+https://github.com/unslothai/unsloth.git" \
    "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"
!pip install -qU "transformers>=4.51,<4.56" accelerate peft trl datasets bitsandbytes
!pip install -qU "torchao>=0.16.0"
!pip install --force-reinstall --no-cache-dir --no-deps "pillow==11.3.0"

# Hot-patch unsloth/models/_utils.py — required.
# Without this, `from unsloth import FastModel` raises:
#     AttributeError: 'NoneType' object has no attribute 'group'
# at unsloth/models/_utils.py:1797:
#     length_spaces = len(re.match(r"[\s]{1,}", BitsAndBytesConfig__init__[0]).group(0))
# That regex assumes BitsAndBytesConfig.__init__'s source starts with ≥1
# whitespace char (class-method indentation). Recent transformers ships
# its dispatched __init__ left-aligned, re.match returns None, .group(0)
# blows up. Swap {1,} → * so a zero-length prefix is valid; length_spaces
# becomes 0 and the dedent loop is a no-op (the right behavior).
# Use importlib.util.find_spec — it locates unsloth without importing it
# (importing it is what's broken).
import importlib.util, pathlib
_spec = importlib.util.find_spec('unsloth')
_utils = pathlib.Path(list(_spec.submodule_search_locations)[0]) / 'models' / '_utils.py'
_src = _utils.read_text()
_old = 're.match(r"[\\s]{1,}", BitsAndBytesConfig__init__[0]).group(0)'
_new = 're.match(r"[\\s]*",   BitsAndBytesConfig__init__[0]).group(0)'
if _old in _src:
    _utils.write_text(_src.replace(_old, _new))
    print(f'✓ patched {_utils} (BitsAndBytesConfig regex)')
elif _new in _src:
    print(f'✓ {_utils} already patched')
else:
    print(f'⚠ {_utils} regex line not found — unsloth shape changed; verify manually')

# Loud assertion: Kaggle base-image upgrades occasionally bump Pillow and
# silently break PIL.Image.thumbnail / convert via ABI mismatch.
import PIL
assert PIL.__version__ == '11.3.0', f'unexpected Pillow version: {PIL.__version__} — pin in install above'

# Same story for torchao — fail loud on next session if Kaggle ships an even
# older one.
import torchao
from packaging.version import Version
assert Version(torchao.__version__) >= Version('0.16.0'), (
    f'torchao {torchao.__version__} is too old — PEFT will fail in get_peft_model'
)

# transformers upper-bound check — defense in depth alongside the hot-patch.
import transformers
assert Version(transformers.__version__) < Version('4.56'), (
    f'transformers {transformers.__version__} too new — re-verify hot-patch above '
    'still applies cleanly to this version of unsloth.'
)
print(f'✓ deps installed (Pillow {PIL.__version__}, torchao {torchao.__version__}, transformers {transformers.__version__})')
print('⚠ if this is your first install in this session, RESTART RUNTIME before running cell 4 (unsloth import)')

In [ ]:
import os, json, time, torch
from pathlib import Path
from unsloth import FastModel
from trl import SFTTrainer, SFTConfig

print(f'CUDA: {torch.cuda.is_available()} | devices: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  [{i}] {torch.cuda.get_device_name(i)} — {p.total_memory / 1e9:.1f} GB')

## 2 — Load Gemma 4 (E2B primary, E4B optional)

Direct HF load (`google/gemma-4-{variant}-it`) — falls back to Unsloth's pre-quantized 4-bit repo if the direct load errors.

In [ ]:
VARIANT = 'e2b'   # 'e2b' (primary, ~2B) or 'e4b' (premium, ~4B)
MAX_SEQ_LEN = 2048

# HF repo IDs use UPPERCASE variant suffix (E2B / E4B).
# Direct:    google/gemma-4-E2B-it
# Fallback:  unsloth/gemma-4-E2B-it-unsloth-bnb-4bit
#   Verify the bnb-4bit repo exists on huggingface.co/unsloth before
#   relying on the fallback. If it 404s, either drop the try/except
#   (load_in_4bit=True on the direct repo handles quantization) or
#   update FALLBACK_REPO to whatever Unsloth published.
DIRECT_REPO   = f'google/gemma-4-{VARIANT.upper()}-it'
FALLBACK_REPO = f'unsloth/gemma-4-{VARIANT.upper()}-it-unsloth-bnb-4bit'

def load_model():
    try:
        print(f'trying direct load: {DIRECT_REPO}')
        return FastModel.from_pretrained(
            model_name=DIRECT_REPO,
            max_seq_length=MAX_SEQ_LEN,
            load_in_4bit=True,
            dtype=None,
        )
    except Exception as e:
        print(f'direct load failed ({type(e).__name__}: {e}) — falling back to {FALLBACK_REPO}')
        return FastModel.from_pretrained(
            model_name=FALLBACK_REPO,
            max_seq_length=MAX_SEQ_LEN,
            load_in_4bit=True,
            dtype=None,
        )

model, tokenizer = load_model()
print(f'✓ loaded {VARIANT.upper()}: {type(model).__name__}')

In [ ]:
model = FastModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    finetune_vision_layers=True,
    finetune_language_layers=True,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
model.print_trainable_parameters()

# Sanity: list any "linear-ish" module the explicit target_modules list
# DOESN'T cover. Gemma 4's vision projector (mm_projector / patch_embed /
# merger) lives outside attention+MLP and won't be LoRA'd unless
# finetune_vision_layers=True actually picks it up. Eyeball the output:
# if you see suspicious untrained projector layers, add them to
# target_modules for the next run.
print('\n--- linear modules NOT in target_modules ---')
covered = {'q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'}
for name, mod in model.named_modules():
    if 'Linear' in type(mod).__name__ and not any(t in name for t in covered):
        if any(k in name.lower() for k in ('proj','embed','merger','vision','mm_')):
            print(f'  {name}  ({type(mod).__name__})')

## 3 — Load dataset (subsampled)

Full dataset is ~190k examples. **v2 retraining**: 40k train / 1k val / 3 epochs (≈ 20–21 h on Kaggle T4, splits across 2 sessions). v1's 5k × 1 epoch was too small to override Gemma 4's reasoning behavior — outputs were hallucinated and didn't follow the structured format. Beyond ~50k samples on a 6-class problem, accuracy gains are <5%.

In [ ]:
from datasets import load_dataset, Image as ImageFeature

# Adjust DATA_ROOT if your Kaggle dataset slug differs
DATA_ROOT   = Path('/kaggle/input/datasets/yaokouadio/pokou-ai-cocoa-training-data')
TRAIN_JSONL = DATA_ROOT / 'train.jsonl'
VAL_JSONL   = DATA_ROOT / 'val.jsonl'

print(f'loading train: {TRAIN_JSONL}')
train_ds = load_dataset('json', data_files=str(TRAIN_JSONL), split='train')
print(f'loading val:   {VAL_JSONL}')
val_ds = load_dataset('json', data_files=str(VAL_JSONL), split='train')
print(f'\nfull dataset:    train={len(train_ds):,}   val={len(val_ds):,}')

# Subsample size — calibrated for the free Kaggle T4 (~5 s/step on Gemma 4
# vision, 12 h kernel limit, 30 h weekly GPU budget). Pick the row that
# matches your time budget; resume_from_checkpoint covers multi-session runs.
#
#   TRAIN_N × NUM_EPOCHS    → steps    → wall time on T4   sessions
#   ─────────────────────────────────────────────────────────────────
#   20_000  × 3            ≈  7.5k     ~10–11 h            1
#   30_000  × 3            ≈ 11.2k     ~15–16 h            2 (split)
#   40_000  × 3            ≈ 15.0k     ~20–21 h            2 (split)
#   60_000  × 2            ≈ 15.0k     ~20–21 h            2 (split)
#
# 40k × 3 is the sweet spot if you commit to a 2-session run — most
# accuracy lift per GPU hour. Diminishing returns above ~50k on a 6-class
# problem. v1 used 5k × 1 epoch; that was the under-training error.
TRAIN_N = 40_000
VAL_N   = 1_000
train_ds = train_ds.shuffle(seed=42).select(range(min(TRAIN_N, len(train_ds))))
val_ds   = val_ds.shuffle(seed=42).select(range(min(VAL_N,   len(val_ds))))
print(f'subsampled:      train={len(train_ds):,}   val={len(val_ds):,}')

In [ ]:
# Build messages with embedded PIL — but lazily.
#
# Unsloth's vision collator expects the actual PIL inside content as
# {"type": "image", "image": <PIL>}. We can't achieve that via .map()
# without serialising N PIL objects into the disk cache (the bug that
# blew the 20 GB Kaggle quota last run).
#
# `set_transform` runs the conversion on every row access — no cache, no
# materialisation. PIL.Image.open is called only when the DataLoader
# fetches a batch, ~8 images per training step.
#
# `thumbnail((896, 896))` caps the long edge at 896 px before the vision
# encoder sees it. Without this, full-resolution augmented images can hang
# the encoder for many minutes per batch.

from PIL import Image as PILImage

MAX_IMG = 896
_bad_rows = 0  # reset per cell run; surface after a few epochs to catch silent corruption


def _extract_or_default(content_list, ctype, key, default=None):
    for c in content_list:
        if c.get('type') == ctype and key in c:
            return c[key]
    return default


def to_chat_transform(batch):
    global _bad_rows
    out_messages = []
    out_images   = []
    for idx, original in enumerate(batch['messages']):
        try:
            image_rel      = _extract_or_default(original[0]['content'], 'image', 'path')
            user_text      = _extract_or_default(original[0]['content'], 'text',  'text', '')
            assistant_text = original[1]['content'][0]['text']
            if image_rel is None:
                raise ValueError('no image path in message[0].content')
            pil = PILImage.open(str(DATA_ROOT / image_rel)).convert('RGB')
            pil.thumbnail((MAX_IMG, MAX_IMG))
            out_messages.append([
                {'role': 'user', 'content': [
                    {'type': 'image', 'image': pil},
                    {'type': 'text',  'text':  user_text},
                ]},
                {'role': 'assistant', 'content': [
                    {'type': 'text', 'text': assistant_text},
                ]},
            ])
            out_images.append(pil)
        except Exception as e:
            _bad_rows += 1
            # Re-raise: silently skipping makes batch shapes inconsistent.
            # If this fires, run a dataset validation pass before retrain.
            raise RuntimeError(f'malformed row at batch idx={idx}: {e}') from e
    return {'messages': out_messages, 'image': out_images}

train_ds.set_transform(to_chat_transform)
val_ds.set_transform(to_chat_transform)

# Sanity check — accesses trigger the lazy transform
sample = train_ds[0]
print(f'columns:          {list(sample.keys())}')
print(f'image type/size:  {type(sample["image"]).__name__} {sample["image"].size}')
print(f'msg[0] role:      {sample["messages"][0]["role"]}')
print(f'msg[0] content[0]: {sample["messages"][0]["content"][0].keys()}')
print(f'\n✓ ready: train={len(train_ds):,}  val={len(val_ds):,}  (PIL loaded lazily, capped at {MAX_IMG}px)')

## 4 — Training config

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator

OUTPUT_DIR = f'/kaggle/working/cocoa_v1_{VARIANT}'

# T4 (compute capability 7.5) does NOT support bf16 — use fp16.
# logging_steps=5 keeps the cell verbose so you can see training is alive.
# save_strategy='steps' + save_steps=200 → LoRA snapshot every 200 steps.
#   With ~7.5k total steps (20k × 3 ep / batch 8) that's ~38 snapshots.
#   save_total_limit=3 keeps disk in check (~90 MB).
# eval_strategy='no' to protect the 12 h kernel budget — vision eval on a
#   1k val set takes ~5 min/epoch and adds up. Run trainer.evaluate()
#   manually after training if you want a final loss number.
# learning_rate=1.5e-4 (down from 2e-4): with 3 epochs the model has more
#   chances to adjust, so a calmer LR avoids overshoot.
# warmup_ratio=0.05 (up from 0.03): slightly longer warmup helps stability.
#   Note: transformers prints a deprecation warning suggesting warmup_steps
#   instead. ratio still works in 5.7 — leaving as-is for portability.
# max_grad_norm=1.0: fp16 vision training without gradient clipping risks
#   loss spikes / NaN. Default in TRL changed across versions — set
#   explicitly.
# report_to='tensorboard': writes scalar event files to OUTPUT_DIR/runs/.
#   Kaggle has TensorBoard support built in (Notebook → ⋮ → TensorBoard).
#   Earlier value 'csv' was a mistake — transformers' Trainer doesn't
#   support csv as a backend. Set to 'none' if you don't need curves.
# Vision-specific: UnslothVisionDataCollator + skip_prepare_dataset.
# dataloader_num_workers=0 to dodge the set_transform / fork hang.

PER_DEVICE_BATCH = 4 if VARIANT == 'e2b' else 2
GRAD_ACCUM       = 2 if VARIANT == 'e2b' else 4
NUM_EPOCHS       = 3

print('training plan:')
print(f'  variant         = {VARIANT}')
print(f'  per-device bsz  = {PER_DEVICE_BATCH}')
print(f'  grad accum      = {GRAD_ACCUM}')
print(f'  effective bsz   = {PER_DEVICE_BATCH * GRAD_ACCUM}')
print(f'  epochs          = {NUM_EPOCHS}')
print(f'  steps/epoch     ≈ {len(train_ds) // (PER_DEVICE_BATCH * GRAD_ACCUM)}')
print(f'  total steps     ≈ {(len(train_ds) // (PER_DEVICE_BATCH * GRAD_ACCUM)) * NUM_EPOCHS}')
print(f'  save every      = 200 steps')
print(f'  output_dir      = {OUTPUT_DIR}')

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=1.5e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        max_grad_norm=1.0,
        optim='adamw_8bit',
        weight_decay=0.01,
        logging_steps=5,
        eval_strategy='no',
        save_strategy='steps',
        save_steps=200,
        save_total_limit=3,
        bf16=False,
        fp16=True,
        max_seq_length=MAX_SEQ_LEN,
        report_to='tensorboard',
        seed=42,
        dataloader_num_workers=0,
        remove_unused_columns=False,
        dataset_kwargs={'skip_prepare_dataset': True},
    ),
)
print('\n✓ trainer initialised (vision collator, single-process loader, save every 200 steps)')

## 5 — Train

In [ ]:
from datetime import datetime

# Checkpoint controls — pick ONE of three modes:
#
#   Mode 1 — fresh training from scratch (default):
#       CHECKPOINT_PATH = None
#       SKIP_TRAINING   = False
#
#   Mode 2 — resume training from a saved checkpoint (multi-session runs):
#       CHECKPOINT_PATH = '/kaggle/input/<dataset>/cocoa_v1_e2b/checkpoint-4200'
#       SKIP_TRAINING   = False
#     Trainer auto-finds the highest-numbered checkpoint inside the path's parent
#     and continues with optimizer + scheduler + RNG state preserved. TRL aborts
#     if VARIANT, dataset, or SFTConfig hyperparams differ from the original run.
#     Workflow: previous session ended → "Save Version (Commit)" persists OUTPUT_DIR
#     into version output → new session attaches that version as Input dataset →
#     point CHECKPOINT_PATH at it.
#
#   Mode 3 — ship the checkpoint as-is, no further training (Path A):
#       CHECKPOINT_PATH = '/kaggle/input/.../cocoa_v1_e2b/checkpoint-4200'
#       SKIP_TRAINING   = True
#     Loads adapter weights into the model and skips trainer.train(). Proceed
#     directly to cell 18 (save merged HF) afterwards. Use this when a session
#     timed out partway through and the checkpoint is "good enough" to ship —
#     e.g., 4200/15000 steps is ~6.7× more training than v1's 625 steps.
CHECKPOINT_PATH = None
SKIP_TRAINING = False

torch.cuda.empty_cache()
print(f'GPU mem before train: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
print(f'starting training at {datetime.now().isoformat(timespec="seconds")}')
print(f'  CHECKPOINT_PATH = {CHECKPOINT_PATH}')
print(f'  SKIP_TRAINING   = {SKIP_TRAINING}\n')

if SKIP_TRAINING:
    if not CHECKPOINT_PATH:
        raise ValueError('SKIP_TRAINING=True requires CHECKPOINT_PATH to be set')
    print(f'Loading LoRA weights from {CHECKPOINT_PATH} (no further training)...')
    import os
    adapter_safe = os.path.join(CHECKPOINT_PATH, 'adapter_model.safetensors')
    adapter_bin  = os.path.join(CHECKPOINT_PATH, 'adapter_model.bin')
    if os.path.exists(adapter_safe):
        from safetensors.torch import load_file
        adapter_state = load_file(adapter_safe)
        print(f'  loaded {len(adapter_state)} tensors from adapter_model.safetensors')
    elif os.path.exists(adapter_bin):
        adapter_state = torch.load(adapter_bin, map_location='cuda')
        print(f'  loaded {len(adapter_state)} tensors from adapter_model.bin')
    else:
        raise FileNotFoundError(
            f'No adapter_model.safetensors or adapter_model.bin in {CHECKPOINT_PATH}'
        )
    _missing, unexpected = model.load_state_dict(adapter_state, strict=False)
    print(f'  unexpected keys: {len(unexpected)} (LoRA tensors only — base weights stay)')
    trainer_stats = None
    print('\n✓ Skipped training. Adapter loaded; proceed to cell 18 to merge + save HF.')
else:
    t0 = time.time()
    trainer_stats = trainer.train(resume_from_checkpoint=CHECKPOINT_PATH or False)
    elapsed = time.time() - t0
    print(f'\n✓ training finished in {elapsed / 60:.1f} min')
    print(f'  final loss      = {trainer_stats.training_loss:.4f}')
    print(f'  total steps     = {trainer_stats.global_step}')
    print(f'  GPU mem peak    = {torch.cuda.max_memory_allocated() / 1e9:.2f} GB')

## 6 — Save LoRA adapter

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

import subprocess
size = subprocess.check_output(['du', '-sh', OUTPUT_DIR]).decode().split()[0]
print(f'✓ LoRA adapter saved to {OUTPUT_DIR} ({size})')

## 7 — Save merged HF model (GGUF conversion done on Mac)

Unsloth's in-Kaggle GGUF converter currently fails on Gemma 4's vision
projector (`unsloth_convert_hf_to_gguf.py --mmproj` returns non-zero).
We save the merged 16-bit HF model here, download it from the version
output, and convert to GGUF Q4_K_M on Mac with a fresh `llama.cpp` build.

Expected size of merged HF: ~5–6 GB. Final GGUF Q4_K_M after Mac-side
conversion: ~1.3–1.6 GB for E2B.

In [ ]:
# Skip in-Kaggle GGUF conversion — Unsloth's bundled llama.cpp converter
# fails on Gemma 4's multimodal vision projector right now.
# Save the merged base+LoRA in HF format instead; convert to GGUF on Mac
# (or via notebook 04) with a fresh llama.cpp build.
#
# Disk hygiene: nuke training checkpoints + unsloth compile cache before
# saving so the merged model fits.
# CUDA hygiene: empty cache + collect garbage before save_pretrained_merged
# loads a fp16 copy of the model. T4 has 16 GB; even ~5 GB of stale
# allocator state can OOM the merge.

import subprocess, gc
!df -h /kaggle/working
!rm -rf {OUTPUT_DIR}/checkpoint-* /kaggle/working/unsloth_compiled_cache
!df -h /kaggle/working

gc.collect()
torch.cuda.empty_cache()
print(f'GPU mem before merge: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

MERGED_DIR = f'/kaggle/working/cocoa_v1_{VARIANT}_merged'
print(f'\nsaving merged model to {MERGED_DIR}...')
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method='merged_16bit')

size = subprocess.check_output(['du', '-sh', MERGED_DIR]).decode().split()[0]
print(f'\n✓ merged model saved ({size})')
!ls -lh {MERGED_DIR}/ | head -20

## 8 — Quick sanity check (single-image inference)

In [ ]:
from transformers import TextStreamer

# val_ds rows have 'messages' (built by to_chat_transform), NOT 'conversations'.
# v1 of this cell used 'conversations' which crashes with KeyError.
sample = val_ds[0]
inputs = tokenizer.apply_chat_template(
    sample['messages'][:1],
    add_generation_prompt=True,
    return_tensors='pt',
    tokenize=True,
).to('cuda')

streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(input_ids=inputs, max_new_tokens=400, streamer=streamer, do_sample=False)

# Optional: post-hoc eval on the full val set (~5 min for 1k vision rows).
# Disabled in-loop to protect the 12 h kernel budget; uncomment once
# training finishes if you want a final loss number.
# results = trainer.evaluate()
# print(f'eval_loss = {results["eval_loss"]:.4f}')

## 9 — Train the other variant

To produce both models for the submission, restart the kernel, set `VARIANT = 'e4b'` in cell 5, and re-run from §3. Outputs land in a separate folder (`cocoa_v1_e4b_gguf/`), so nothing is overwritten.